In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-sonnet-4-5-20250929"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [3]:
file_metadata = upload("streaming.csv")
file_metadata

FileMetadata(id='file_011CaoKNG6iuQdJ5e12eU6mr', created_at=datetime.datetime(2026, 5, 7, 13, 51, 18, 247000, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

In [4]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
                Run a detailed analysis to determine major drivers of churn.
                Your final output should include at least one detailed plot summarizing your findings.

                Critical note: Every time you execute code, you're starting with a completely clean slate. 
                No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

response = chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])
response

Message(id='msg_01PZ3URJaGqbewSoRHtHeTTK', container=Container(id='container_011CaoKNJxczLxHgSFRpzVt7', expires_at=datetime.datetime(2026, 5, 7, 14, 56, 40, 589485, tzinfo=TzInfo(0))), content=[TextBlock(citations=None, text="I'll help you run a detailed churn analysis on the streaming.csv file. Let me start by exploring the data and then perform a comprehensive analysis to identify the major drivers of churn.", type='text'), ServerToolUseBlock(id='srvtoolu_01MCg21YxwCut6yKwrNywVye', caller=None, input={'command': 'cd $INPUT_DIR && head -20 streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSE

In [5]:
response

Message(id='msg_01PZ3URJaGqbewSoRHtHeTTK', container=Container(id='container_011CaoKNJxczLxHgSFRpzVt7', expires_at=datetime.datetime(2026, 5, 7, 14, 56, 40, 589485, tzinfo=TzInfo(0))), content=[TextBlock(citations=None, text="I'll help you run a detailed churn analysis on the streaming.csv file. Let me start by exploring the data and then perform a comprehensive analysis to identify the major drivers of churn.", type='text'), ServerToolUseBlock(id='srvtoolu_01MCg21YxwCut6yKwrNywVye', caller=None, input={'command': 'cd $INPUT_DIR && head -20 streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSE

In [6]:
len(response.content)

42

In [44]:
for index, block in enumerate(response.content):
    if block.type == "bash_code_execution_tool_result":
        print(block)

BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSER_00002,Premium,41.4,Drama,5,9,45.7,3,17.99,0\nUSER_00003,Standard,33.6,Action,1,7,32.3,4,12.99,1\nUSER_00004,Standard,115.6,Action,12,33,57.3,1,12.99,0\nUSER_00005,Basic,93.8,Documentary,9,27,90.0,2,7.99,1\nUSER_00006,Basic,105.6,Romance,10,27,80.0,2,7.99,0\nUSER_00007,Basic,106.6,Thriller,8,23,53.1,3,7.99,0\nUSER_00008,Premium,93.6,Documentary,11,27,55.1,2,17.99,0\nUSER_00009,Standard,138.9,Comedy,12,36,86.4,1,12.99,0\nUSER_00010,Standard,56.9,Action,6,17,56.2,2,12.99,0\nUSER_00011,Basic,24.5,Horror,5,9,38.9,4,7.99,1\nUSER_00012,Premium,117.5,Documentary,11,25,75.5,1,17.99,0\nUSER_00013,Standard,109.2,Docume

In [ ]:
for index, block in enumerate(response.content):
    if block.type == "text":
        print(block.text)
    if block.type == "server_tool_use":
        print(f"{block.input["file_text"] if hasattr(block.input, 'file_text') else {block.input["command"][::]}}")  
    if block.type == "text_editor_code_execution_tool_result":
        print(f"{block}")

I'll help you run a detailed churn analysis on the streaming.csv file. Let me start by exploring the data and then perform a comprehensive analysis to identify the major drivers of churn.
{'cd $INPUT_DIR && head -20 streaming.csv'}
Now let me perform a comprehensive churn analysis:
{'create'}
TextEditorCodeExecutionToolResultBlock(content=TextEditorCodeExecutionCreateResultBlock(is_file_update=False, type='text_editor_code_execution_create_result'), tool_use_id='srvtoolu_01ArhzUFPY7zENXf4vwiM46r', type='text_editor_code_execution_tool_result')
{'cd /tmp && python churn_analysis.py'}
Now let me create a comprehensive visualization summarizing the findings:
{'create'}
TextEditorCodeExecutionToolResultBlock(content=TextEditorCodeExecutionCreateResultBlock(is_file_update=False, type='text_editor_code_execution_create_result'), tool_use_id='srvtoolu_01JCmyE8cgrGkLe1SduheMgY', type='text_editor_code_execution_tool_result')
{'cd /tmp && python create_visualizations.py'}
Let me fix the code er

In [ ]:
# BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=
#                                                                       [BashCodeExecutionOutputBlock(file_id='file_011CaoKjTqFPvwjSiad8GLYg', type='bash_code_execution_output'), 
#                                                                        BashCodeExecutionOutputBlock(file_id='file_011CaoKjWw3PzCo8HHDWveJX', type='bash_code_execution_output'), 
#                                                                        BashCodeExecutionOutputBlock(file_id='file_011CaoKjZvNrScHua7pDLpEA', type='bash_code_execution_output'), 
#                                                                        BashCodeExecutionOutputBlock(file_id='file_011CaoKjcfKp58sBUULsdaxd', type='bash_code_execution_output')], return_code=0, stderr='', stdout='✅ Files exported to OUTPUT_DIR:\ntotal 2.3M\n-rw-r--r-- 1 root root 4.2K May  7 13:56 analysis_summary.txt\n-rw-r--r-- 1 root root 1.6M May  7 13:56 churn_analysis_dashboard.png\n-rw-r--r-- 1 root root 658K May  7 13:56 churn_insights_summary.png\n-rw-r--r-- 1 root root  12K May  7 13:56 executive_summary.txt\n', type='bash_code_execution_result'), tool_use_id='srvtoolu_01QtdpCvVi4ZcCEpActMdTnX', type='bash_code_execution_tool_result')


In [41]:
download_file("file_011CaoKjcfKp58sBUULsdaxd")